<h1 style="color: royalblue; text-align:center">Data Loading</br>

---

## Imports

In [6]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [7]:
import pandas as pd
import csv
from collections import Counter
from src.parser import parse_gzip,filter_proteins

## Consts

In [8]:
DATA_PATH = "../data/uniprot_sprot.dat.gz"
OUTPUT_PATH = "../data/mini_cafa.csv"

MAX_PROTEINS = 20000
MAX_SEQUENCE_LENGTH = 1000
TOP_N_GO_TERMS = 20

proteins = []

## Loading

In [9]:
label_counter = Counter()

In [10]:
for entry in parse_gzip(DATA_PATH):
    print(entry)
    break

{'accession': 'Q6GZX4', 'sequence': 'MAFSAEDVLKEYDRRRRMEALLLSLYYPNDRKLLDYKEWSPPRVQVECPKAPVEWNNPPSEKGLIVGHFSGIKYKGEKAQASEVDVNKMCCWVSKFKDAMRRYQGIQTCKIPGKVLSDLDAKIKAYNLTVEGVEGFVRYSRVTKQHVAAFLKELRHSKQYENVNLIHYILTDKRVDIQHLEKDLVKDFKALVESAHRMRQGHMINVKYILYQLLKKHGHGPDGPDILTVKTGSKGVLYDDSFRKIYTDLGWKFTPL', 'go_terms': ['GO:0046782']}


In [11]:
for entry in parse_gzip(DATA_PATH):
    if entry['go_terms'] and entry['sequence']:
        proteins.append(entry)
        label_counter.update(entry['go_terms'])
    
    if len(proteins) >= MAX_PROTEINS:
        break

In [12]:
print(f"Total number of proteins: {len(proteins)}")


Total number of proteins: 20000


In [13]:
filtered_proteins = filter_proteins(proteins, max_sequence_length=MAX_SEQUENCE_LENGTH)
print(f"Number of proteins after filtering: {len(filtered_proteins)}")

Number of proteins after filtering: 18929


In [ ]:
top_labels = set(
    label for label, count in label_counter.most_common(TOP_N_GO_TERMS)
)

cleaned = []

for p in filtered_proteins:
    filtered_labels = [x for x in p["go_terms"] if x in top_labels]
    
    if len(filtered_labels) == 0:
        continue
    
    cleaned.append({
        "accession": p["accession"],
        "sequence": p["sequence"],
        "seq_len": len(p["sequence"]),
        "labels": ";".join(filtered_labels)
    })

df = pd.DataFrame(cleaned)

print(df.head())
print("Final dataset size:", len(df))
df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

print("\nTop labels:")
for label, count in label_counter.most_common(TOP_N_GO_TERMS):
    print(label, count)

  accession                                           sequence  seq_len  \
0    Q6GZW6  MDTSPYDFLKLYPWLSRGEADKGTLLDAFPGETFEQSLASDVAMRR...      948   
1    Q6GZV6  MATNYCDEFERNPTRNPRTGRTIKRGGPVFRALERECSDGAARVFP...      851   
2    Q91G63  MKQNLLILLSLLLVVVAIMWWLYEKKKEVPLPPPTPPTPPTPTGVP...      134   
3    Q197B6  MPLSVFAEEFAEKSVKRYIGQGLWLPCNLSDYYYYQEFHDEGGYGS...      335   
4    Q91G50  MDKIEELKIEELKIEIPQRKTKFFHDSENSDKRDEEETLNPTITSK...      123   

                  labels  
0  GO:0005524;GO:0003677  
1             GO:0005524  
2             GO:0016020  
3             GO:0005524  
4             GO:0016020  
Final dataset size: 15255
Saved to ../data/mini_cafa.csv

Top labels:
GO:0005524 4018
GO:0005737 3764
GO:0005829 3463
GO:0005886 2762
GO:0046872 1840
GO:0006633 1773
GO:0005576 1742
GO:0005634 1676
GO:0016020 1571
GO:0008270 1505
GO:0000287 1051
GO:0003989 1008
GO:2001295 1007
GO:0016743 986
GO:0006508 901
GO:0090729 814
GO:0005739 794
GO:0009317 777
GO:0003677 684
GO:0016887 634
